In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis
                            )
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from lifelines import CoxPHFitter
from sklearn.preprocessing import StandardScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
response_OUS.isna().sum().sum()

0

In [4]:
# Need to choose patient_id from OUS_D3 in response_OUS
data = list(OUS_D3['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D3 with response_OUS
clinical_train = pd.merge(OUS_D3, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [5]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [6]:
# Check null values in D3
clinical_train.isnull().sum().sum()

0

In [7]:
clinical_train

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,event_OS
0,54.238356,1,0,1,0,0,1,0.0,0,0.000000,...,0.029974,0.815962,0.000000,0.002035,0.140311,0.000062,0.009991,0.000370,114.312329,0
1,54.539726,0,0,0,0,1,0,0.0,1,27.404795,...,0.059728,0.707300,0.000000,0.008732,0.191058,0.000349,0.023402,0.000699,111.846575,0
2,59.019178,0,1,0,0,0,1,0.0,1,41.019178,...,0.039463,0.819828,0.000034,0.002518,0.126531,0.000000,0.009452,0.000414,49.479452,1
3,70.726027,0,0,0,0,1,0,0.0,1,37.500000,...,0.061133,0.716212,0.000000,0.003296,0.192388,0.000000,0.018879,0.000899,17.589041,1
4,67.865753,0,0,0,0,1,0,0.0,1,53.000000,...,0.058589,0.696493,0.000199,0.008968,0.202073,0.000399,0.029892,0.001395,66.739726,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,60.435616,0,0,1,0,0,1,1.0,0,0.000000,...,0.029444,0.804831,0.000000,0.001442,0.152626,0.000000,0.009734,0.000361,42.805479,0
135,68.794521,0,0,1,0,0,1,1.0,0,0.000000,...,0.046451,0.792151,0.000000,0.004094,0.142778,0.000079,0.011810,0.000630,42.739726,0
136,57.498630,0,0,1,0,0,1,1.0,1,39.498630,...,0.016316,0.832202,0.000000,0.000914,0.140582,0.000000,0.009137,0.000522,42.312329,0
137,65.684932,0,0,1,0,0,1,1.0,1,71.527397,...,0.038862,0.787588,0.000000,0.003144,0.156640,0.000054,0.011978,0.000705,42.246575,0


## Test dataset: MAASTRO 

In [8]:
(MAASTRO_D3['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [9]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [10]:
rows_with_nan = response_MAASTRO[response_MAASTRO.isna().any(axis=1)]

print("Rows with NaN values:")
print(rows_with_nan)

Rows with NaN values:
    patient_id  OS  OS_event  LRC  LRC_event  DFS  DFS_event
10          11 NaN       NaN  NaN        NaN  NaN        NaN
20          21 NaN       NaN  NaN        NaN  NaN        NaN
31          32 NaN       NaN  NaN        NaN  NaN        NaN
51          52 NaN       NaN  NaN        NaN  NaN        NaN
83          84 NaN       NaN  NaN        NaN  NaN        NaN
86          87 NaN       NaN  NaN        NaN  NaN        NaN


In [11]:
# Assuming your data is stored in a list or a pandas DataFrame/Series
# Convert data to a set for faster membership checking
data_set = set(list(response_MAASTRO['patient_id']))

# Find numbers from 1 to 114 not present in data
missing_numbers = set(range(1, 115)) - data_set

# Convert missing_numbers back to a sorted list
missing_numbers_list = sorted(missing_numbers)

print("Numbers from 1 to 114 not present in data:", missing_numbers_list)

Numbers from 1 to 114 not present in data: []


In [12]:
MAASTRO_D3

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,...,LBP_003_PET,LBP_012_PET,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET
0,1,55,0,0,1,0,0,1,1,1,...,0.000026,0.001181,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341
1,2,55,0,0,1,0,0,0,0,0,...,0.000056,0.002735,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335
2,3,55,0,0,1,0,0,0,0,1,...,0.000286,0.001888,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229
3,4,61,1,0,0,0,1,1,0,1,...,0.000160,0.003037,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240
4,6,70,0,0,1,0,0,1,1,1,...,0.000071,0.000881,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,...,0.000000,0.000544,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078
95,111,63,0,0,0,0,1,0,0,1,...,0.000109,0.000869,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489
96,112,63,0,0,1,0,0,1,1,1,...,0.000000,0.000734,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141
97,113,54,0,0,1,0,0,1,1,0,...,0.000114,0.001640,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038


In [13]:
# Assuming your data is stored in a list or a pandas DataFrame/Series
# Convert data to a set for faster membership checking
data_set = set(list(MAASTRO_D3['patient_id']))

# Find numbers from 1 to 114 not present in data
missing_numbers = set(range(1, 115)) - data_set

# Convert missing_numbers back to a sorted list
missing_numbers_list = sorted(missing_numbers)

print("Numbers from 1 to 114 not present in data:", missing_numbers_list)


Numbers from 1 to 114 not present in data: [5, 9, 11, 21, 32, 36, 46, 52, 65, 76, 78, 81, 84, 87, 92]


In [14]:
# Assuming your data is stored in a list or a pandas DataFrame/Series
# Convert data to a set for faster membership checking
data_set = set(list(response_MAASTRO['patient_id']))

# Find numbers from 1 to 114 not present in data
missing_numbers = set(range(1, 115)) - data_set

# Convert missing_numbers back to a sorted list
missing_numbers_list = sorted(missing_numbers)

print("Numbers from 1 to 114 not present in data:", missing_numbers_list)


Numbers from 1 to 114 not present in data: []


In [15]:
# need to choose patient_id from MAASTRO_D3 in response_MAASTRO
data = list(MAASTRO_D3['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 

In [16]:
# Merge MAASTRO_D3 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D3, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]

In [17]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [18]:
# Check if some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event


In [19]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [20]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]
lower, upper = np.percentile(y_MAASTRO['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

X_train:  (139, 388)
y_train:  (139,)


(99, 390)

# Feature Selection: RENT

In [21]:
selected_features = ["hpv_related",
                    "uicc8_III-IV",
                    "shape_Sphericity",
                    "oropharynx",
                    "LBP_102_PET",
                    "pack_years"]



# Selecting features in the DataFrame
X_rent = X[selected_features]
X_new = X_rent.copy()

X_MAASTRO_rent = X_MAASTRO[selected_features]
MAASTRO_new = X_MAASTRO_rent.copy()

# Standardization

In [22]:
original_X = X.copy()

In [23]:
# Standardize X_new, the new data with the selected features only 
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = StandardScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [24]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns

MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [25]:
X_new

,hpv_related,uicc8_III-IV,shape_Sphericity,oropharynx,LBP_102_PET,pack_years
0,0.0,0.0,0.761164,1,0.000000,0.000000
1,0.0,0.0,0.697049,0,0.000000,27.404795
2,0.0,1.0,0.565792,0,0.000034,41.019178
3,0.0,0.0,0.684364,0,0.000000,37.500000
4,0.0,0.0,0.503142,0,0.000199,53.000000
...,...,...,...,...,...,...
134,1.0,0.0,0.742102,1,0.000000,0.000000
135,1.0,1.0,0.722918,1,0.000000,0.000000
136,1.0,0.0,0.652963,1,0.000000,39.498630
137,1.0,1.0,0.724255,1,0.000000,71.527397


In [26]:
X_new_std

,hpv_related,uicc8_III-IV,shape_Sphericity,oropharynx,LBP_102_PET,pack_years
0,0.0,0.0,1.075042,1,-0.562251,-1.101176
1,0.0,0.0,0.242767,0,-0.562251,0.105775
2,0.0,1.0,-1.461057,0,-0.202838,0.705374
3,0.0,0.0,0.078112,0,-0.562251,0.550384
4,0.0,0.0,-2.274305,0,1.514080,1.233028
...,...,...,...,...,...,...
134,1.0,0.0,0.827589,1,-0.562251,-1.101176
135,1.0,1.0,0.578577,1,-0.562251,-1.101176
136,1.0,0.0,-0.329498,1,-0.562251,0.638406
137,1.0,1.0,0.595927,1,-0.562251,2.049005


In [27]:
MAASTRO_new

,hpv_related,uicc8_III-IV,shape_Sphericity,oropharynx,LBP_102_PET,pack_years
0,1,0,0.668072,1,0.000026,0
1,0,1,0.669961,1,0.000167,20
2,0,1,0.624081,1,0.000057,6
3,0,1,0.577624,0,0.000000,45
4,1,0,0.630933,1,0.000000,59
...,...,...,...,...,...,...
94,0,1,0.671754,0,0.000000,55
95,0,1,0.632189,0,0.000000,174
96,1,1,0.645548,1,0.000000,0
97,1,0,0.727488,1,0.000000,0


In [28]:
MAASTRO_new_std

,hpv_related,uicc8_III-IV,shape_Sphericity,oropharynx,LBP_102_PET,pack_years
0,1,0,-0.133381,1,-0.288893,-1.101176
1,0,1,-0.108854,1,1.182201,-0.220344
2,0,1,-0.704414,1,0.033974,-0.836927
3,0,1,-1.307469,0,-0.562251,0.880696
4,1,0,-0.615466,1,-0.562251,1.497278
...,...,...,...,...,...,...
94,0,1,-0.085581,0,-0.562251,1.321112
95,0,1,-0.599161,0,-0.562251,6.562062
96,1,1,-0.425762,1,-0.562251,-1.101176
97,1,0,0.637891,1,-0.562251,-1.101176


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [29]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-19 17:17:21,973] A new study created in memory with name: no-name-7d81af19-0852-44ff-a95f-0b5cd557f74e


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.8354978354978355
Fold 2 C-index: 0.7946428571428571
Fold 3 C-index: 0.7745098039215687


[I 2024-04-19 17:17:34,596] A new study created in memory with name: no-name-d809fb08-bd8b-4d41-bf01-b23afdbeb9e4


Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 17:17:34,548] Trial 0 finished with value: 0.7737571035318616 and parameters: {}. Best is trial 0 with value: 0.7737571035318616.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7737571035318616], datetime_start=datetime.datetime(2024, 4, 19, 17, 17, 22, 106445), datetime_complete=datetime.datetime(2024, 4, 19, 17, 17, 34, 547851), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7737571035318616


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.13733677912072229
Fold 2 IBS: 0.17153895698332644
Fold 3 IBS: 0.1822486408327174
Fold 4 IBS: 0.18669684505649792
Fold 5 IBS: 0.2297802490038966
[I 2024-04-19 17:17:35,469] Trial 0 finished with value: 0.1815202941994321 and parameters: {}. Best is trial 0 with value: 0.1815202941994321.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.1815202941994321], datetime_start=datetime.datetime(2024, 4, 19, 17, 17, 34, 635172), datetime_complete=datetime.datetime(2024, 4, 19, 17, 17, 35, 469439), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.1815202941994321


In [30]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [31]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.774
train_ibs:  0.182


#### Test

In [32]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [33]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.587
IBS score: 0.278


In [34]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [35]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge 

#### Train

In [36]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 17:17:35,819] A new study created in memory with name: no-name-057cd4da-226a-454b-a29f-a8af5af1f250


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6861471861471862
Fold 2 C-index: 0.7544642857142857
Fold 3 C-index: 0.7769607843137255


[I 2024-04-19 17:17:36,252] A new study created in memory with name: no-name-a42aa055-2e81-457c-9f55-82edae71a109


Fold 4 C-index: 0.6645569620253164
Fold 5 C-index: 0.7089201877934272
[I 2024-04-19 17:17:36,239] Trial 0 finished with value: 0.7182098811987881 and parameters: {}. Best is trial 0 with value: 0.7182098811987881.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7182098811987881], datetime_start=datetime.datetime(2024, 4, 19, 17, 17, 35, 876345), datetime_complete=datetime.datetime(2024, 4, 19, 17, 17, 36, 239553), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7182098811987881


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397651532980308
Fold 2 IBS: 0.22157790808722821
Fold 3 IBS: 0.20453594046886417
Fold 4 IBS: 0.22473803633656894
Fold 5 IBS: 0.21812430978731348
[I 2024-04-19 17:17:36,875] Trial 0 finished with value: 0.2165905420019556 and parameters: {}. Best is trial 0 with value: 0.2165905420019556.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2165905420019556], datetime_start=datetime.datetime(2024, 4, 19, 17, 17, 36, 324443), datetime_complete=datetime.datetime(2024, 4, 19, 17, 17, 36, 875782), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2165905420019556


In [37]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [38]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.718
train_ibs:  0.217


#### Test

In [39]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [40]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.538


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [41]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [42]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 17:17:37,335] A new study created in memory with name: no-name-85b531ad-7137-4d91-ad25-5ececdb92e36


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.7974683544303798


[I 2024-04-19 17:17:38,555] A new study created in memory with name: no-name-42946897-b0f9-44fb-97b7-9d5ffd84535c


Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 17:17:38,543] Trial 0 finished with value: 0.7730663726940719 and parameters: {}. Best is trial 0 with value: 0.7730663726940719.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7730663726940719], datetime_start=datetime.datetime(2024, 4, 19, 17, 17, 37, 380573), datetime_complete=datetime.datetime(2024, 4, 19, 17, 17, 38, 542947), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7730663726940719


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.1379815599301658
Fold 2 IBS: 0.17215662611010385
Fold 3 IBS: 0.18053265337011185
Fold 4 IBS: 0.18575880946622436
Fold 5 IBS: 0.22805949185504237
[I 2024-04-19 17:17:39,836] Trial 0 finished with value: 0.18089782814632965 and parameters: {}. Best is trial 0 with value: 0.18089782814632965.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.18089782814632965], datetime_start=datetime.datetime(2024, 4, 19, 17, 17, 38, 600130), datetime_complete=datetime.datetime(2024, 4, 19, 17, 17, 39, 836308), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.18089782814632965


In [43]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [44]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.773
train_ibs:  0.181


#### Test 

In [45]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [46]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.589


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.273


In [47]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [48]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 17:17:40,515] A new study created in memory with name: no-name-19c4ac76-a513-4839-b753-61f5cf4a170e


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 17:17:41,496] Trial 0 finished with value: 0.7730663726940719 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7730663726940719.
Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 17:17:42,584] Trial 1 finished with value: 0.771193123394352 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.7730663726940719.
Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 17:17:43,632] Trial 2 finished with value: 0.7721735155512147 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with v

Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 17:18:07,448] Trial 24 finished with value: 0.7730663726940719 and parameters: {'l1_ratio': 0.7602371709740536}. Best is trial 0 with value: 0.7730663726940719.
Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 17:18:08,439] Trial 25 finished with value: 0.7730663726940719 and parameters: {'l1_ratio': 0.891573855041803}. Best is trial 0 with value: 0.7730663726940719.
Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 17:18:09,479] Trial 26 finished with value: 0.7720859805372091 and parameters: {'l1_ratio': 0.4514399534035586}.

Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 17:18:35,004] Trial 48 finished with value: 0.7720859805372091 and parameters: {'l1_ratio': 0.5033132688848968}. Best is trial 0 with value: 0.7730663726940719.
Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 17:18:36,214] Trial 49 finished with value: 0.7721735155512147 and parameters: {'l1_ratio': 0.355745098499332}. Best is trial 0 with value: 0.7730663726940719.
Fold 1 C-index: 0.8311688311688312
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 17:18:37,290] Trial 50 finished with value: 0.7730663726940719 and parameters: {'l1_ratio': 0.8449632453631055}. Best is tr

...

Fold 1 IBS: 0.13793637425831798
Fold 2 IBS: 0.17183494234012317
Fold 3 IBS: 0.18049836494084173
Fold 4 IBS: 0.18581535873060714
Fold 5 IBS: 0.22800104577515204
[I 2024-04-19 17:19:36,059] Trial 0 finished with value: 0.1808172172090084 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.1808172172090084.
Fold 1 IBS: 0.13793532645260137
Fold 2 IBS: 0.1728542315875333
Fold 3 IBS: 0.18050715395021202
Fold 4 IBS: 0.18582222546343835
Fold 5 IBS: 0.22799275555239545
[I 2024-04-19 17:19:37,389] Trial 1 finished with value: 0.1810223386012361 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.1808172172090084.
Fold 1 IBS: 0.13793971259714122
Fold 2 IBS: 0.17296754027310474
Fold 3 IBS: 0.18042264582960602
Fold 4 IBS: 0.18588828708244048
Fold 5 IBS: 0.22789958641884048
[I 2024-04-19 17:19:38,783] Trial 2 finished with value: 0.1810235544402266 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.180817217209008

Fold 1 IBS: 0.1379634226175415
Fold 2 IBS: 0.17179969264781605
Fold 3 IBS: 0.18053559923697263
Fold 4 IBS: 0.18577970117611475
Fold 5 IBS: 0.2280337281339789
[I 2024-04-19 17:20:09,356] Trial 25 finished with value: 0.18082242876248478 and parameters: {'l1_ratio': 0.6515450395903271}. Best is trial 22 with value: 0.18079863958544093.
Fold 1 IBS: 0.13793444081901537
Fold 2 IBS: 0.17213688296063961
Fold 3 IBS: 0.1804791869215465
Fold 4 IBS: 0.18581786631407746
Fold 5 IBS: 0.2279669781264248
[I 2024-04-19 17:20:10,430] Trial 26 finished with value: 0.18086707102834074 and parameters: {'l1_ratio': 0.5288665517746662}. Best is trial 22 with value: 0.18079863958544093.
Fold 1 IBS: 0.1379520613481779
Fold 2 IBS: 0.17204607949482273
Fold 3 IBS: 0.18052425412115375
Fold 4 IBS: 0.18579666164607517
Fold 5 IBS: 0.22804430048925287
[I 2024-04-19 17:20:11,479] Trial 27 finished with value: 0.18087267141989646 and parameters: {'l1_ratio': 0.9147109952383484}. Best is trial 22 with value: 0.1807986395

Fold 5 IBS: 0.21785203675459358
[I 2024-04-19 17:20:41,044] Trial 49 finished with value: 0.21626053920865135 and parameters: {'l1_ratio': 0.005040986123852953}. Best is trial 41 with value: 0.18077662800177344.
Fold 1 IBS: 0.13792713199403955
Fold 2 IBS: 0.17180488723302254
Fold 3 IBS: 0.18054107280277185
Fold 4 IBS: 0.1858270095305308
Fold 5 IBS: 0.22804439783732694
[I 2024-04-19 17:20:42,326] Trial 50 finished with value: 0.18082889987953835 and parameters: {'l1_ratio': 0.7132558744331859}. Best is trial 41 with value: 0.18077662800177344.
Fold 1 IBS: 0.13795353854176148
Fold 2 IBS: 0.17176603625833156
Fold 3 IBS: 0.18041959050306167
Fold 4 IBS: 0.18579268794361894
Fold 5 IBS: 0.22791750319792375
[I 2024-04-19 17:20:43,420] Trial 51 finished with value: 0.18076987128893948 and parameters: {'l1_ratio': 0.6673101903985772}. Best is trial 51 with value: 0.18076987128893948.
Fold 1 IBS: 0.13795107762303802
Fold 2 IBS: 0.17177423544866502
Fold 3 IBS: 0.18043089087756187
Fold 4 IBS: 0.185

Fold 1 IBS: 0.13797579647200553
Fold 2 IBS: 0.17180503535269107
Fold 3 IBS: 0.18048348642018877
Fold 4 IBS: 0.18576351447997327
Fold 5 IBS: 0.22798038337905935
[I 2024-04-19 17:21:11,674] Trial 74 finished with value: 0.1808016432207836 and parameters: {'l1_ratio': 0.6327893114683547}. Best is trial 54 with value: 0.1807697924860232.
Fold 1 IBS: 0.13797122345242527
Fold 2 IBS: 0.17185895853692826
Fold 3 IBS: 0.18045360327605411
Fold 4 IBS: 0.18576992952226384
Fold 5 IBS: 0.22795901683284717
[I 2024-04-19 17:21:12,925] Trial 75 finished with value: 0.18080254632410372 and parameters: {'l1_ratio': 0.7417557707331788}. Best is trial 54 with value: 0.1807697924860232.
Fold 1 IBS: 0.13795495068028404
Fold 2 IBS: 0.17224615305956753
Fold 3 IBS: 0.18055117699939435
Fold 4 IBS: 0.1857914453652373
Fold 5 IBS: 0.22787601114340608
[I 2024-04-19 17:21:14,056] Trial 76 finished with value: 0.18088394744957786 and parameters: {'l1_ratio': 0.5043196387796003}. Best is trial 54 with value: 0.180769792

Fold 1 IBS: 0.13795648252467935
Fold 2 IBS: 0.1717556493971905
Fold 3 IBS: 0.1804060830269236
Fold 4 IBS: 0.18578881461687277
Fold 5 IBS: 0.22806471828399183
[I 2024-04-19 17:21:42,023] Trial 99 finished with value: 0.18079434956993162 and parameters: {'l1_ratio': 0.6625385041189411}. Best is trial 95 with value: 0.1807662508357407.


* Best trial for IBS: 
 FrozenTrial(number=95, state=TrialState.COMPLETE, values=[0.1807662508357407], datetime_start=datetime.datetime(2024, 4, 19, 17, 21, 36, 135888), datetime_complete=datetime.datetime(2024, 4, 19, 17, 21, 37, 249794), params={'l1_ratio': 0.6650774161075232}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=None)}, trial_id=95, value=None)


* Best Score for IBS: 
 0.1807662508357407


In [49]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [50]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.773
train_ibs:  0.181


#### Test

In [51]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [52]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.6964995386793018)

test_cindex : 0.589


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.6650774161075232)

test_ibs:  0.273


In [53]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [54]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-19 17:21:42,722] A new study created in memory with name: no-name-57e2b7bb-3c59-4278-836d-e567e2c58e5d


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.8268398268398268
Fold 2 C-index: 0.7232142857142857
Fold 3 C-index: 0.7941176470588235
Fold 4 C-index: 0.7088607594936709
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 17:21:52,611] Trial 0 finished with value: 0.7439398371546546 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7439398371546546.
Fold 1 C-index: 0.8138528138528138
Fold 2 C-index: 0.7098214285714286
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.7637130801687764
Fold 5 C-index: 0.6995305164319249
[I 2024-04-19 17:22:00,258] Trial 1 finished with value: 0.7591482736873416 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 

Fold 5 C-index: 0.7417840375586855
[I 2024-04-19 17:24:26,658] Trial 15 finished with value: 0.8120274196000707 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 7, 'n_estimators': 480, 'oob_score': True, 'max_samples': 0.4470849120697906, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.08807762154599924, 'warm_start': True}. Best is trial 15 with value: 0.8120274196000707.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 17:24:37,472] Trial 16 finished with value: 0.5 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 7, 'min_samples_leaf': 1, 'max_depth': 7, 'n_estimators': 498, 'oob_score': True, 'max_samples': 0.36871324569404207, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.20791692251205332, 'warm_start': True}. Best is trial 15 with value: 0.8120274196000707.
Fold 1 C-index: 0.8181818181818182
Fold 2 C-index: 0.8125
Fold 3 C-index: 0.83333333333

Fold 3 C-index: 0.7965686274509803
Fold 4 C-index: 0.770042194092827
Fold 5 C-index: 0.7652582159624414
[I 2024-04-19 17:26:29,590] Trial 30 finished with value: 0.7825128767653189 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 6, 'min_samples_leaf': 6, 'max_depth': 1, 'n_estimators': 225, 'oob_score': True, 'max_samples': 0.5856012699634701, 'max_features': None, 'min_weight_fraction_leaf': 0.029048761893088054, 'warm_start': True}. Best is trial 19 with value: 0.8265809184741336.
Fold 1 C-index: 0.8268398268398268
Fold 2 C-index: 0.8303571428571429
Fold 3 C-index: 0.8284313725490197
Fold 4 C-index: 0.810126582278481
Fold 5 C-index: 0.755868544600939
[I 2024-04-19 17:26:36,910] Trial 31 finished with value: 0.810324693825082 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 4, 'min_samples_leaf': 11, 'max_depth': 4, 'n_estimators': 339, 'oob_score': True, 'max_samples': 0.5365082685042595, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.0654723504402752, 'wa

...

Fold 5 IBS: 0.2171431998129361
[I 2024-04-19 17:36:02,606] Trial 31 finished with value: 0.17792400393817337 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 17, 'min_samples_leaf': 6, 'max_depth': 4, 'n_estimators': 310, 'oob_score': False, 'max_samples': 0.6683617295863028, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.11208498314187895}. Best is trial 12 with value: 0.17673667335392534.
Fold 1 IBS: 0.14711946655783312
Fold 2 IBS: 0.19006811981061228
Fold 3 IBS: 0.158033991341854
Fold 4 IBS: 0.17821885585463562
Fold 5 IBS: 0.21747675580865514
[I 2024-04-19 17:36:13,414] Trial 32 finished with value: 0.17818343787471802 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 20, 'min_samples_leaf': 8, 'max_depth': 5, 'n_estimators': 310, 'oob_score': False, 'max_samples': 0.6782402761258088, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.10884764482583958}. Best is trial 12 with value: 0.17673667335392534.
Fold 1 IBS: 0.1474328186504624
Fold 2 IBS: 0.1925

Fold 1 IBS: 0.16189795049200065
Fold 2 IBS: 0.2002316521231612
Fold 3 IBS: 0.1632441421719029
Fold 4 IBS: 0.17861160833008888
Fold 5 IBS: 0.21900022601901006
[I 2024-04-19 17:37:55,018] Trial 47 finished with value: 0.18459711582723273 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 4, 'max_depth': 1, 'n_estimators': 130, 'oob_score': True, 'max_samples': 0.5217274207107006, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.0488148159561139}. Best is trial 38 with value: 0.17659645958662992.
Fold 1 IBS: 0.15250696953387413
Fold 2 IBS: 0.1814915271847653
Fold 3 IBS: 0.16047524829592633
Fold 4 IBS: 0.17246364113740187
Fold 5 IBS: 0.2120974321417849
[I 2024-04-19 17:37:58,673] Trial 48 finished with value: 0.1758069636587505 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 18, 'min_samples_leaf': 10, 'max_depth': 2, 'n_estimators': 70, 'oob_score': False, 'max_samples': 0.5834369548668846, 'max_features': 'sqrt', 'min_weight_fraction_leaf

Fold 1 IBS: 0.1528522544145544
Fold 2 IBS: 0.1883938076545454
Fold 3 IBS: 0.16059370160325126
Fold 4 IBS: 0.17447839059286202
Fold 5 IBS: 0.21675029793203757
[I 2024-04-19 17:38:48,400] Trial 63 finished with value: 0.17861369043945013 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 6, 'min_samples_leaf': 11, 'max_depth': 6, 'n_estimators': 121, 'oob_score': False, 'max_samples': 0.5082465499234735, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.0848024624668733}. Best is trial 48 with value: 0.1758069636587505.
Fold 1 IBS: 0.16166621880208873
Fold 2 IBS: 0.1731632023212488
Fold 3 IBS: 0.1700340266068216
Fold 4 IBS: 0.17478280891532624
Fold 5 IBS: 0.2135771701385966
[I 2024-04-19 17:38:49,642] Trial 64 finished with value: 0.17864468535681638 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 5, 'min_samples_leaf': 14, 'max_depth': 4, 'n_estimators': 16, 'oob_score': False, 'max_samples': 0.588577514842217, 'max_features': 'sqrt', 'min_weight_fraction_leaf':

Fold 1 IBS: 0.14805872849452492
Fold 2 IBS: 0.1844394959428906
Fold 3 IBS: 0.1592084284475042
Fold 4 IBS: 0.18049902936533177
Fold 5 IBS: 0.21718793826468474
[I 2024-04-19 17:39:25,291] Trial 79 finished with value: 0.17787872410298725 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 12, 'min_samples_leaf': 10, 'max_depth': 11, 'n_estimators': 50, 'oob_score': False, 'max_samples': 0.8221486447141937, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.10873595340811325}. Best is trial 48 with value: 0.1758069636587505.
Fold 1 IBS: 0.15361491788372222
Fold 2 IBS: 0.19642745280621465
Fold 3 IBS: 0.16095616456284817
Fold 4 IBS: 0.22665346294232952
Fold 5 IBS: 0.2385594621067867
[I 2024-04-19 17:39:27,337] Trial 80 finished with value: 0.19524229206038027 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 41, 'oob_score': False, 'max_samples': 0.7659640622029704, 'max_features': None, 'min_weight_fraction_leaf

Fold 5 IBS: 0.21759783800196422
[I 2024-04-19 17:40:15,967] Trial 94 finished with value: 0.1776435259600116 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 12, 'min_samples_leaf': 9, 'max_depth': 5, 'n_estimators': 119, 'oob_score': False, 'max_samples': 0.7242593013645032, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.14462066268912804}. Best is trial 81 with value: 0.17538981813433546.
Fold 1 IBS: 0.15162608620389867
Fold 2 IBS: 0.19121869268171396
Fold 3 IBS: 0.15836366628343707
Fold 4 IBS: 0.17826802865577213
Fold 5 IBS: 0.21920067113279704
[I 2024-04-19 17:40:21,319] Trial 95 finished with value: 0.17973542899152378 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 8, 'max_depth': 4, 'n_estimators': 137, 'oob_score': False, 'max_samples': 0.772556078583092, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.1631127999140187}. Best is trial 81 with value: 0.17538981813433546.
Fold 1 IBS: 0.16352525950573188
Fold 2 IBS: 0.1829

In [55]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [56]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.871
train_ibs:  0.175


#### Test

In [57]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [58]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=14, max_features='auto', max_leaf_nodes=15,
                     max_samples=0.9504102015210659, min_samples_leaf=2,
                     min_samples_split=9,
                     min_weight_fraction_leaf=0.005622697057305325,
                     n_estimators=144, random_state=123, warm_start=True)

test_cindex:  0.618


RandomSurvivalForest(max_depth=6, max_leaf_nodes=10,
                     max_samples=0.5785272262773831, min_samples_leaf=8,
                     min_samples_split=11,
                     min_weight_fraction_leaf=0.08332907721465231,
                     random_state=123)

test_ibs:  0.223


In [59]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [60]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [61]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 17:40:40,657] A new study created in memory with name: no-name-afe11036-8e5e-4646-bb00-25a3266ec7ae


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.8225108225108225
Fold 2 C-index: 0.7589285714285714
Fold 3 C-index: 0.8333333333333334
Fold 4 C-index: 0.8016877637130801
Fold 5 C-index: 0.6666666666666666
[I 2024-04-19 17:40:43,909] Trial 0 finished with value: 0.7766254315304948 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7766254315304948.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 17:40:50,960] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.8181818181818182
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.8161764705882353
Fold 4 C-index: 0.7932489451476793
Fold 5 C-index: 0.6572769953051644
[I 2024-04-19 17:42:23,241] Trial 16 finished with value: 0.7660839887017223 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.6535355702555379, 'min_weight_fraction_leaf': 0.1906011147608997}. Best is trial 12 with value: 0.786055160528368.
Fold 1 C-index: 0.8333333333333334
Fold 2 C-index: 0.7834821428571429
Fold 3 C-index: 0.8063725490196079
Fold 4 C-index: 0.8080168776371308
Fold 5 C-index: 0.6314553990610329
[I 2024-04-19 17:42:26,793] Trial 17 finished with value: 0.7725320603816497 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 10, 'n_estimators': 384, 'oob_score': False, 'warm_start': True, 'max_featur

Fold 1 C-index: 0.8268398268398268
Fold 2 C-index: 0.7723214285714286
Fold 3 C-index: 0.8284313725490197
Fold 4 C-index: 0.8227848101265823
Fold 5 C-index: 0.676056338028169
[I 2024-04-19 17:43:19,138] Trial 31 finished with value: 0.7852867552230053 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 11, 'min_samples_leaf': 3, 'max_depth': 11, 'n_estimators': 204, 'oob_score': False, 'warm_start': True, 'max_features': 1, 'max_samples': 0.8438862458923003, 'min_weight_fraction_leaf': 0.03902180415717787}. Best is trial 25 with value: 0.7919233215012451.
Fold 1 C-index: 0.8181818181818182
Fold 2 C-index: 0.7723214285714286
Fold 3 C-index: 0.8284313725490197
Fold 4 C-index: 0.8185654008438819
Fold 5 C-index: 0.676056338028169
[I 2024-04-19 17:43:21,823] Trial 32 finished with value: 0.7827112716348635 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 12, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 283, 'oob_score': False, 'warm_start': True, 'max_features':

...

Fold 1 IBS: 0.1585529260071
Fold 2 IBS: 0.218153010449042
Fold 3 IBS: 0.17021896151952115
Fold 4 IBS: 0.16343124711768475
Fold 5 IBS: 0.2311751417053347
[I 2024-04-19 19:02:19,166] Trial 30 finished with value: 0.18830625735973652 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 19, 'min_samples_leaf': 6, 'max_depth': 3, 'n_estimators': 116, 'oob_score': False, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.8258632202024414, 'min_weight_fraction_leaf': 0.20241101766143602}. Best is trial 7 with value: 0.1857272253521116.
Fold 1 IBS: 0.15420880459120728
Fold 2 IBS: 0.22137976081415967
Fold 3 IBS: 0.16774275391760834
Fold 4 IBS: 0.15688252726853427
Fold 5 IBS: 0.23380379197752338
[I 2024-04-19 19:02:23,022] Trial 31 finished with value: 0.18680352771380657 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 17, 'min_samples_leaf': 7, 'max_depth': 17, 'n_estimators': 108, 'oob_score': False, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.61

Fold 1 IBS: 0.1350712935921733
Fold 2 IBS: 0.21575795396259614
Fold 3 IBS: 0.1719895889282686
Fold 4 IBS: 0.1639662174932425
Fold 5 IBS: 0.23179177468347978
[I 2024-04-19 19:06:30,790] Trial 45 finished with value: 0.18371536573195205 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 6, 'min_samples_leaf': 2, 'max_depth': 8, 'n_estimators': 446, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.7218458884001862, 'min_weight_fraction_leaf': 0.002908146688295732}. Best is trial 42 with value: 0.18220424087625745.
Fold 1 IBS: 0.14405258172603636
Fold 2 IBS: 0.23510729165077407
Fold 3 IBS: 0.1652427330079092
Fold 4 IBS: 0.15610840831496184
Fold 5 IBS: 0.23462655467653304
[I 2024-04-19 19:06:48,802] Trial 46 finished with value: 0.1870275138752429 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 4, 'min_samples_leaf': 2, 'max_depth': 12, 'n_estimators': 302, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.77768969

Fold 1 IBS: 0.2139754605574802
Fold 2 IBS: 0.22128037493850963
Fold 3 IBS: 0.2049770044390806
Fold 4 IBS: 0.22468785016066256
Fold 5 IBS: 0.21820838836670314
[I 2024-04-19 19:11:02,058] Trial 60 finished with value: 0.21662581569248723 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 3, 'min_samples_leaf': 4, 'max_depth': 9, 'n_estimators': 373, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.36612707136592737, 'min_weight_fraction_leaf': 0.2583710069782479}. Best is trial 42 with value: 0.18220424087625745.
Fold 1 IBS: 0.13013980993149055
Fold 2 IBS: 0.21254887098310069
Fold 3 IBS: 0.17298441817166582
Fold 4 IBS: 0.16679488873081466
Fold 5 IBS: 0.2331102795997352
[I 2024-04-19 19:11:19,938] Trial 61 finished with value: 0.18311565348336137 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 7, 'min_samples_leaf': 2, 'max_depth': 8, 'n_estimators': 439, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.77533595

Fold 1 IBS: 0.1331805487459175
Fold 2 IBS: 0.21624338660783993
Fold 3 IBS: 0.1727837835092345
Fold 4 IBS: 0.16512709360909275
Fold 5 IBS: 0.23118212185240014
[I 2024-04-19 19:15:29,035] Trial 75 finished with value: 0.18370338686489696 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 7, 'min_samples_leaf': 2, 'max_depth': 11, 'n_estimators': 388, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.8812725582045702, 'min_weight_fraction_leaf': 0.014257699585348107}. Best is trial 42 with value: 0.18220424087625745.
Fold 1 IBS: 0.14741275991375746
Fold 2 IBS: 0.22339111860096061
Fold 3 IBS: 0.16719852475388572
Fold 4 IBS: 0.1582361988967138
Fold 5 IBS: 0.23150079709187427
[I 2024-04-19 19:15:49,966] Trial 76 finished with value: 0.18554787985143836 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 8, 'min_samples_leaf': 1, 'max_depth': 10, 'n_estimators': 446, 'oob_score': True, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.96

Fold 1 IBS: 0.12988717417010312
Fold 2 IBS: 0.21111739221808115
Fold 3 IBS: 0.17281260544267968
Fold 4 IBS: 0.16744968489433068
Fold 5 IBS: 0.23530465742353276
[I 2024-04-19 19:19:43,703] Trial 90 finished with value: 0.18331430282974548 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 10, 'n_estimators': 317, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.7774148609541289, 'min_weight_fraction_leaf': 0.011005982719496389}. Best is trial 42 with value: 0.18220424087625745.
Fold 1 IBS: 0.12951620582034926
Fold 2 IBS: 0.210867217186133
Fold 3 IBS: 0.17327356003962616
Fold 4 IBS: 0.16765928672732763
Fold 5 IBS: 0.2351767398201044
[I 2024-04-19 19:19:57,086] Trial 91 finished with value: 0.1832986019187081 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 10, 'n_estimators': 322, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.77818

In [62]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [63]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.816
train_ibs:  0.181


#### Test

In [64]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [65]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=11, max_leaf_nodes=13,
                   max_samples=0.9554816381741302, min_samples_leaf=1,
                   min_samples_split=17,
                   min_weight_fraction_leaf=0.026333169142875475,
                   n_estimators=68, random_state=123, warm_start=True)

C-index score: 0.656


ExtraSurvivalTrees(max_depth=13, max_features=1, max_leaf_nodes=11,
                   max_samples=0.7098772062937373, min_samples_leaf=1,
                   min_samples_split=14,
                   min_weight_fraction_leaf=0.012799049735992758,
                   n_estimators=288, oob_score=True, random_state=123,
                   warm_start=True)

IBS: 0.214


In [66]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [67]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-19 19:21:39,891] A new study created in memory with name: no-name-6543ff49-8c6d-4a21-9da5-a4b8c636bb05


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 19:22:50,452] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 19:23:27,442] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 19:43:48,388] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7471201964999683.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 19:46:17,150] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 20:14:31,370] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 22 with value: 0.7520782207516631.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5558035714285714
Fold 3 C-index: 0.6666666666666666
Fold 4 C-index: 0.5654008438818565
Fold 5 C-index: 0.636150234741784
[I 2024-04-19 20:17:37,344] Trial 26 finished with value: 0.5848042633437757 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropo

...

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609563
[I 2024-04-19 22:25:42,692] Trial 55 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.918589848048704, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.1366452847323028, 'n_estimators': 388, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.18967222528443176, 'max_features': 'auto', 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 15, 'max_leaf_nodes': 15, 'min_samples_leaf': 19, 'max_depth': 18}. Best is trial 53 with value: 0.21558143837167032.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609582
[I 2024-04-19 22:26:25,180] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.7513175598857118, 'learning_rate': 0.0985092209

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-19 22:32:32,152] Trial 66 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9998769156045215, 'learning_rate': 0.05359198006915804, 'dropout_rate': 0.650983074619535, 'n_estimators': 500, 'criterion': 'squared_error', 'ccp_alpha': 0.713342732410399, 'min_weight_fraction_leaf': 0.037327349410482574, 'max_features': 0.1, 'min_impurity_decrease': 4.636270239719423e-07, 'validation_fraction': 0.7909756969364712, 'min_samples_split': 9, 'max_leaf_nodes': 8, 'min_samples_leaf': 18, 'max_depth': 19}. Best is trial 64 with value: 0.21551873707634944.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.2181243152560957
[I 2024-04-19 22:32:59,852] Trial 67 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8945047973351204, 'learning_rate': 0.03028968218872250

Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-19 22:38:28,643] Trial 77 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.6527746876723795, 'learning_rate': 0.009617814274368723, 'dropout_rate': 0.23464360454432553, 'n_estimators': 449, 'criterion': 'squared_error', 'ccp_alpha': 0.5872217111247551, 'min_weight_fraction_leaf': 0.12767889602965304, 'max_features': 1, 'min_impurity_decrease': 1.4509408420175117e-05, 'validation_fraction': 0.9291496186906082, 'min_samples_split': 8, 'max_leaf_nodes': 6, 'min_samples_leaf': 20, 'max_depth': 5}. Best is trial 72 with value: 0.21520875346396268.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609557
[I 2024-04-19 22:39:17,416] Trial 78 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9412481314247185, 'learning_rate': 0.01401320577538

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-19 22:45:38,386] Trial 88 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.7762725560789899, 'learning_rate': 0.011111179381597284, 'dropout_rate': 0.3196243460718578, 'n_estimators': 407, 'criterion': 'squared_error', 'ccp_alpha': 1.3457840140168202, 'min_weight_fraction_leaf': 0.026102976372990007, 'max_features': 0.1, 'min_impurity_decrease': 4.1842504540759167e-07, 'validation_fraction': 0.8468844412711507, 'min_samples_split': 5, 'max_leaf_nodes': 20, 'min_samples_leaf': 7, 'max_depth': 8}. Best is trial 72 with value: 0.21520875346396268.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-19 22:46:03,579] Trial 89 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.6969649633096852, 'learning_rate': 0.01535870401

Fold 3 IBS: 0.2023614521860876
Fold 4 IBS: 0.2231898890507258
Fold 5 IBS: 0.2170482819994517
[I 2024-04-19 22:52:08,214] Trial 99 finished with value: 0.2148526864616871 and parameters: {'subsample': 0.8333920238412017, 'learning_rate': 0.02538174390008161, 'dropout_rate': 0.4577965489047534, 'n_estimators': 399, 'criterion': 'squared_error', 'ccp_alpha': 0.004367047248614303, 'min_weight_fraction_leaf': 0.18072249935794454, 'max_features': 'auto', 'min_impurity_decrease': 4.472271308488144e-07, 'validation_fraction': 0.9580200141092102, 'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 14, 'max_depth': 5}. Best is trial 99 with value: 0.2148526864616871.


* Best trial for IBS: 
 FrozenTrial(number=99, state=TrialState.COMPLETE, values=[0.2148526864616871], datetime_start=datetime.datetime(2024, 4, 19, 22, 51, 45, 872936), datetime_complete=datetime.datetime(2024, 4, 19, 22, 52, 8, 213523), params={'subsample': 0.8333920238412017, 'learning_rate': 0.02538174390008161, 

In [68]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [69]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.791
train_ibs:  0.215


#### Test

In [70]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [71]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.030433464476024484,
                                 criterion='squared_error',
                                 dropout_rate=0.6934300990357392,
                                 learning_rate=0.05396213117321635, max_depth=7,
                                 max_features=1, max_leaf_nodes=10,
                                 min_impurity_decrease=3.7676942671090066e-07,
                                 min_samples_leaf=11, min_samples_split=20,
                                 min_weight_fraction_leaf=0.38729753243593296,
                                 n_estimators=424, random_state=123,
                                 subsample=0.38604321258984187,
                                 validation_fraction=0.9213195792266449)

C-index score: 0.63


GradientBoostingSurvivalAnalysis(ccp_alpha=0.004367047248614303,
                                 criterion='squared_error',
                                 dropout_rate=0.4577965489047534,
                                 learning_rate=0.02538174390008161, max_depth=5,
                                 max_features='auto', max_leaf_nodes=17,
                                 min_impurity_decrease=4.472271308488144e-07,
                                 min_samples_leaf=14, min_samples_split=3,
                                 min_weight_fraction_leaf=0.18072249935794454,
                                 n_estimators=399, random_state=123,
                                 subsample=0.8333920238412017,
                                 validation_fraction=0.9580200141092102)

IBS: 0.22


In [72]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [73]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [74]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 22:52:16,503] A new study created in memory with name: no-name-af458b58-ce7c-4e9f-aa54-ed9f9d1df126


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7352941176470589
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-19 22:52:17,099] Trial 0 finished with value: 0.7136894859079793 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.7136894859079793.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7352941176470589
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-19 22:52:21,975] Trial 1 finished with value: 0.7136894859079793 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.7136894859079793.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7352941176470589
Fold 4 C-index: 0.6624472573839663
Fold 5 C-ind

Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.7401960784313726
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7230046948356808
[I 2024-04-19 22:53:05,110] Trial 19 finished with value: 0.7118990866496844 and parameters: {'subsample': 0.37351945568220557, 'dropout_rate': 0.5732915269816466, 'n_estimators': 285, 'learning_rate': 0.08697733590757546}. Best is trial 12 with value: 0.7199992328573758.
Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.7544642857142857
Fold 3 C-index: 0.7450980392156863
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7183098591549296
[I 2024-04-19 22:53:08,450] Trial 20 finished with value: 0.7145920268219121 and parameters: {'subsample': 0.1868200799888373, 'dropout_rate': 0.7859448591953863, 'n_estimators': 419, 'learning_rate': 0.06551866052750378}. Best is trial 12 with value: 0.7199992328573758.
Fold 1 C-index: 0.6926406926406926
Fold 2 C-index: 0.7589285714285714
Fold 3 C-index: 0.7549019607843137


Fold 1 C-index: 0.7012987012987013
Fold 2 C-index: 0.7544642857142857
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.679324894514768
Fold 5 C-index: 0.704225352112676
[I 2024-04-19 22:53:43,105] Trial 38 finished with value: 0.7188430388849489 and parameters: {'subsample': 0.1485025056401248, 'dropout_rate': 0.13272164755980653, 'n_estimators': 267, 'learning_rate': 0.06976187125928615}. Best is trial 22 with value: 0.722579853713313.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7352941176470589
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-19 22:53:45,537] Trial 39 finished with value: 0.7136894859079793 and parameters: {'subsample': 0.608751577387902, 'dropout_rate': 0.20477224083522255, 'n_estimators': 326, 'learning_rate': 0.07571568533625589}. Best is trial 22 with value: 0.722579853713313.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.7352941176470589
Fold 4 C-index: 0.

...

Fold 5 IBS: 0.19031531086295966
[I 2024-04-19 22:56:49,868] Trial 19 finished with value: 0.18985831471526118 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9930703343219778, 'n_estimators': 51, 'learning_rate': 0.04096883532946153}. Best is trial 19 with value: 0.18985831471526118.
Fold 1 IBS: 0.19158806255701005
Fold 2 IBS: 0.18704819923805763
Fold 3 IBS: 0.1815437858617923
Fold 4 IBS: 0.2077856758404389
Fold 5 IBS: 0.19501411232740778
[I 2024-04-19 22:56:50,099] Trial 20 finished with value: 0.19259596716494137 and parameters: {'subsample': 0.1435040576337751, 'dropout_rate': 0.7933084651006226, 'n_estimators': 43, 'learning_rate': 0.04127909023805986}. Best is trial 19 with value: 0.18985831471526118.
Fold 1 IBS: 0.19048310390879777
Fold 2 IBS: 0.19146711563365057
Fold 3 IBS: 0.18443716068386018
Fold 4 IBS: 0.20603449857822337
Fold 5 IBS: 0.19676257062338176
[I 2024-04-19 22:56:50,293] Trial 21 finished with value: 0.19383688988558273 and parameters: {'subsamp

Fold 1 IBS: 0.19425199130724533
Fold 2 IBS: 0.18919788026883572
Fold 3 IBS: 0.1846687129175461
Fold 4 IBS: 0.2095154755197313
Fold 5 IBS: 0.19106686432630335
[I 2024-04-19 22:57:00,228] Trial 39 finished with value: 0.19374018486793237 and parameters: {'subsample': 0.3645779440068705, 'dropout_rate': 0.9451944095204212, 'n_estimators': 32, 'learning_rate': 0.055414173921640594}. Best is trial 19 with value: 0.18985831471526118.
Fold 1 IBS: 0.19382347350487195
Fold 2 IBS: 0.192810194009499
Fold 3 IBS: 0.18739980802076284
Fold 4 IBS: 0.2076512731118766
Fold 5 IBS: 0.19621051262343964
[I 2024-04-19 22:57:00,495] Trial 40 finished with value: 0.19557905225408997 and parameters: {'subsample': 0.4306300092933272, 'dropout_rate': 0.27073406810212547, 'n_estimators': 48, 'learning_rate': 0.028415559992356662}. Best is trial 19 with value: 0.18985831471526118.
Fold 1 IBS: 0.19695915226740335
Fold 2 IBS: 0.1871233493208149
Fold 3 IBS: 0.18359408934098623
Fold 4 IBS: 0.21307946438079317
Fold 5 IB

Fold 2 IBS: 0.2645632795089968
Fold 3 IBS: 0.22894090424448954
Fold 4 IBS: 0.27552911976963496
Fold 5 IBS: 0.23319463968568588
[I 2024-04-19 22:57:08,606] Trial 58 finished with value: 0.25278637994865194 and parameters: {'subsample': 0.21054619864137353, 'dropout_rate': 0.8779252987509731, 'n_estimators': 136, 'learning_rate': 0.06399320755529825}. Best is trial 19 with value: 0.18985831471526118.
Fold 1 IBS: 0.1951331515625371
Fold 2 IBS: 0.18636834675674185
Fold 3 IBS: 0.17814609892988456
Fold 4 IBS: 0.22004746150957186
Fold 5 IBS: 0.18596510105775765
[I 2024-04-19 22:57:08,897] Trial 59 finished with value: 0.1931320319632986 and parameters: {'subsample': 0.10168115254416726, 'dropout_rate': 0.9277275476862245, 'n_estimators': 61, 'learning_rate': 0.05881817053907151}. Best is trial 19 with value: 0.18985831471526118.
Fold 1 IBS: 0.19816597395300953
Fold 2 IBS: 0.19055593340574914
Fold 3 IBS: 0.1792043707554762
Fold 4 IBS: 0.23078723076382143
Fold 5 IBS: 0.18861364253468677
[I 2024

Fold 3 IBS: 0.18789694550460964
Fold 4 IBS: 0.2094878237035773
Fold 5 IBS: 0.20271223192541332
[I 2024-04-19 22:57:21,453] Trial 78 finished with value: 0.19879813975869523 and parameters: {'subsample': 0.21909833876880208, 'dropout_rate': 0.8421552729690653, 'n_estimators': 10, 'learning_rate': 0.09824860662987817}. Best is trial 61 with value: 0.18940035391035645.
Fold 1 IBS: 0.19795467948012913
Fold 2 IBS: 0.18686789007694735
Fold 3 IBS: 0.18176135837016866
Fold 4 IBS: 0.215728572537313
Fold 5 IBS: 0.1875320056789117
[I 2024-04-19 22:57:21,626] Trial 79 finished with value: 0.19396890122869398 and parameters: {'subsample': 0.25048469619790803, 'dropout_rate': 0.9972287781621585, 'n_estimators': 24, 'learning_rate': 0.09149284816474539}. Best is trial 61 with value: 0.18940035391035645.
Fold 1 IBS: 0.2477049966054357
Fold 2 IBS: 0.2468750702523289
Fold 3 IBS: 0.2160310038823229
Fold 4 IBS: 0.27445405024970665
Fold 5 IBS: 0.22346811144460804
[I 2024-04-19 22:57:22,067] Trial 80 finish

Fold 1 IBS: 0.2241667356223695
Fold 2 IBS: 0.21223858109275415
Fold 3 IBS: 0.19498546542031991
Fold 4 IBS: 0.25591406954107443
Fold 5 IBS: 0.1993076467080594
[I 2024-04-19 22:57:27,498] Trial 98 finished with value: 0.21732249967691547 and parameters: {'subsample': 0.23364862132974257, 'dropout_rate': 0.21013178546832284, 'n_estimators': 75, 'learning_rate': 0.07957653523690165}. Best is trial 61 with value: 0.18940035391035645.
Fold 1 IBS: 0.18305240938707887
Fold 2 IBS: 0.18553060319162343
Fold 3 IBS: 0.17766413974978526
Fold 4 IBS: 0.20051843386987528
Fold 5 IBS: 0.1949576105515627
[I 2024-04-19 22:57:27,684] Trial 99 finished with value: 0.1883446393499851 and parameters: {'subsample': 0.12637473058568036, 'dropout_rate': 0.10237467903157496, 'n_estimators': 26, 'learning_rate': 0.08679980373545465}. Best is trial 99 with value: 0.1883446393499851.


* Best trial for IBS: 
 FrozenTrial(number=99, state=TrialState.COMPLETE, values=[0.1883446393499851], datetime_start=datetime.dateti

In [75]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [76]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.723
train_ibs:  0.188


#### Test

In [77]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [78]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.984207766523338,
                                              learning_rate=0.08046846831573967,
                                              n_estimators=446,
                                              random_state=123,
                                              subsample=0.10604166872611101)

C-index score: 0.555


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.10237467903157496,
                                              learning_rate=0.08679980373545465,
                                              n_estimators=26, random_state=123,
                                              subsample=0.12637473058568036)

IBS: 0.226


In [79]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [80]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.871,1.0
ExtraSurvivalTrees,0.816,2.0
GradientBoosting,0.791,3.0
CoxPH,0.774,4.0
CoxLasso,0.773,5.5
CoxElastic,0.773,5.5
ComponentwiseGradientBoosting,0.723,7.0
CoxRidge,0.718,8.0


In [81]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.175,1.0
CoxLasso,0.181,3.0
CoxElastic,0.181,3.0
ExtraSurvivalTrees,0.181,3.0
CoxPH,0.182,5.0
ComponentwiseGradientBoosting,0.188,6.0
GradientBoosting,0.215,7.0
CoxRidge,0.217,8.0


In [82]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
ExtraSurvivalTrees,0.656,1.0
GradientBoosting,0.630,2.0
Randomsurvivalforest,0.618,3.0
CoxLasso,0.589,4.5
CoxElastic,0.589,4.5
CoxPH,0.587,6.0
ComponentwiseGradientBoosting,0.555,7.0
CoxRidge,0.538,8.0


In [83]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs 

,IBS,rank
ExtraSurvivalTrees,0.214,1.0
GradientBoosting,0.220,2.0
CoxRidge,0.221,3.0
Randomsurvivalforest,0.223,4.0
ComponentwiseGradientBoosting,0.226,5.0
CoxLasso,0.273,6.5
CoxElastic,0.273,6.5
CoxPH,0.278,8.0


In [84]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d3/os/standard/rent/' 

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d3_os_standard_rent_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [85]:
from datetime import date

current_date = date.today()
print(current_date)

2024-04-19
